In [3]:
# ============================================================
# PACKAGE 9.3 : SHAP EXPLAINABILITY (CORRECTED)
# ============================================================

import pandas as pd
import numpy as np

import shap
import joblib

import matplotlib.pyplot as plt


print("="*75)
print("PACKAGE 9.3 : SHAP EXPLAINABILITY")
print("="*75)


# ------------------------------------------------------------
# Load Test Data
# ------------------------------------------------------------

X_train = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package4_Feature_Selection\X_train_selected.csv"
)


X_test = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package4_Feature_Selection\X_test_selected.csv"
)


y_test = pd.read_csv(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Package3_Feature_Encoding\y_test.csv"
).squeeze()



print("\nDataset Shapes")

print("X_train :", X_train.shape)

print("X_test  :", X_test.shape)

print("y_test  :", y_test.shape)



# ------------------------------------------------------------
# Load XGBoost Model
# ------------------------------------------------------------

print("\nLoading Trained Model...")


model = joblib.load(
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Forecasting Model Pipeline\Baseline_RF_X\XGBoost\XGBoost_AQI_Model.pkl"
)


print("Model Loaded Successfully")



# ------------------------------------------------------------
# SHAP Explainer
# ------------------------------------------------------------

print("\nCalculating SHAP Values...")


explainer = shap.TreeExplainer(
    model
)


shap_values = explainer.shap_values(
    X_test
)


print("SHAP Calculation Completed")



# ------------------------------------------------------------
# Save SHAP Values
# ------------------------------------------------------------

shap_df = pd.DataFrame(
    shap_values,
    columns=X_test.columns
)


shap_df.to_csv(
    "SHAP_Values.csv",
    index=False
)



# ------------------------------------------------------------
# Global Feature Importance
# ------------------------------------------------------------

feature_importance = pd.DataFrame(
    {

        "Feature":
        X_test.columns,


        "Mean_SHAP_Value":
        np.abs(shap_values).mean(axis=0)

    }
)



feature_importance = (
    feature_importance
    .sort_values(
        by="Mean_SHAP_Value",
        ascending=False
    )
)



print("\n============================================================")
print("TOP 20 SHAP FEATURES")
print("============================================================")


print(
    feature_importance.head(20)
)



feature_importance.to_csv(
    "SHAP_Feature_Importance.csv",
    index=False
)



# ------------------------------------------------------------
# CORRECT SATELLITE FEATURE DETECTION
# ------------------------------------------------------------

satellite_keywords = [

    "AOD",
    "MODIS",
    "SENTINEL",
    "S5P",
    "TROPOMI"

]


satellite_features = [

    feature
    for feature in X_test.columns
    if any(
        key.lower() in feature.lower()
        for key in satellite_keywords
    )

]


print("\n============================================================")

if len(satellite_features) > 0:

    print(
        "Satellite Features Found:"
    )

    for feature in satellite_features:
        print("-", feature)

else:

    print(
        "No Satellite Features Found"
    )


print("============================================================")



# ------------------------------------------------------------
# SHAP BAR PLOT
# ------------------------------------------------------------

plt.figure()

shap.summary_plot(
    shap_values,
    X_test,
    plot_type="bar",
    show=False
)


plt.tight_layout()

plt.savefig(
    "SHAP_Bar.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# ------------------------------------------------------------
# SHAP BEESWARM
# ------------------------------------------------------------

plt.figure()

shap.summary_plot(
    shap_values,
    X_test,
    show=False
)


plt.tight_layout()

plt.savefig(
    "SHAP_Beeswarm.png",
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# ------------------------------------------------------------
# PM2.5 Dependence Plot
# ------------------------------------------------------------

if "PM2.5" in X_test.columns:

    shap.dependence_plot(

        "PM2.5",

        shap_values,

        X_test,

        show=False

    )


    plt.tight_layout()

    plt.savefig(
        "SHAP_Dependence_PM25.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()



# ------------------------------------------------------------
# Satellite AOD Dependence Plot
# ------------------------------------------------------------

aod_features = [

    f for f in X_test.columns
    if "AOD" in f

]


if len(aod_features) > 0:


    shap.dependence_plot(

        aod_features[0],

        shap_values,

        X_test,

        show=False

    )


    plt.tight_layout()

    plt.savefig(
        "SHAP_Dependence_AOD.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.close()



# ------------------------------------------------------------
# LOCAL EXPLANATION
# ------------------------------------------------------------

sample_index = 0


shap.plots._waterfall.waterfall_legacy(

    explainer.expected_value,

    shap_values[sample_index],

    X_test.iloc[sample_index],

    show=False

)


plt.tight_layout()


plt.savefig(

    "SHAP_Waterfall.png",

    dpi=300,

    bbox_inches="tight"

)


plt.close()



# ------------------------------------------------------------
# Final Output
# ------------------------------------------------------------

print("\n============================================================")
print("FILES SAVED")
print("============================================================")


print("1. SHAP_Feature_Importance.csv")
print("2. SHAP_Beeswarm.png")
print("3. SHAP_Bar.png")
print("4. SHAP_Dependence_PM25.png")
print("5. SHAP_Dependence_AOD.png")
print("6. SHAP_Waterfall.png")
print("7. SHAP_Values.csv")


print("\nPACKAGE 9.3 COMPLETED SUCCESSFULLY")

PACKAGE 9.3 : SHAP EXPLAINABILITY

Dataset Shapes
X_train : (1826, 30)
X_test  : (182, 30)
y_test  : (182,)

Loading Trained Model...
Model Loaded Successfully

Calculating SHAP Values...
SHAP Calculation Completed

TOP 20 SHAP FEATURES
            Feature  Mean_SHAP_Value
0               AQI        38.558350
1             PM2.5        35.547840
2              PM10         9.151375
6           AQI_MA3         8.089654
12          AQI_MA7         6.561667
3          Rainfall         5.336049
5         PM10_Diff         4.916412
9               NOx         4.397388
4   Humidity_Change         4.152227
10               CO         2.840474
8       Wind_Change         2.796906
25         AQI_Lag1         2.750369
15        PM25_Diff         2.272799
14        MODIS_AOD         1.832244
29    MODIS_AOD_MA7         1.793763
7               NO2         1.686579
11               NO         1.640249
23         PM25_MA3         1.578753
18        Wind_Lag1         1.407711
24         AQI_Lag7    